# TriageIQ - 03 DistilBERT vs Baseline
Fine-tune DistilBERT on a subsample (CPU-friendly) and compare macro-F1, latency, and size against the TF-IDF + LR baseline.

Prereqs: `pip install torch transformers`, `02_baseline` already run (creates `../models/baseline.joblib`). First run downloads the model (~250 MB).

## 1. Load data and split (same time-based split as the baseline)

In [ ]:
import pandas as pd, numpy as np, torch, time, joblib
from sklearn.metrics import f1_score, classification_report
df = pd.read_csv("../data/311_requests.csv", parse_dates=["created_date"])
df["description"] = df["description"].fillna("").str.lower().str.strip()
cut = pd.Timestamp("2025-10-01")
train, test = df[df.created_date < cut], df[df.created_date >= cut].copy()
train = train.sample(4000, random_state=42)
classes = sorted(df["category"].unique()); c2i = {c:i for i,c in enumerate(classes)}
print(len(train), len(test), classes)

## 2. Tokenize

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
NAME = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(NAME)
def encode(texts): return tok(list(texts), truncation=True, padding=True, max_length=48, return_tensors="pt")
Xtr, Xte = encode(train["description"]), encode(test["description"])
ytr = torch.tensor(train["category"].map(c2i).values)
yte = test["category"].map(c2i).values
print(Xtr["input_ids"].shape)

## 3. Fine-tune (plain PyTorch loop)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSequenceClassification.from_pretrained(NAME, num_labels=len(classes)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=5e-5)
EPOCHS, BS = 2, 32
n = len(ytr)
for ep in range(EPOCHS):
    model.train(); perm = torch.randperm(n); total = 0
    for i in range(0, n, BS):
        idx = perm[i:i+BS]
        batch = {k: v[idx].to(device) for k, v in Xtr.items()}
        out = model(**batch, labels=ytr[idx].to(device))
        opt.zero_grad(); out.loss.backward(); opt.step()
        total += out.loss.item() * len(idx)
    print(f"epoch {ep+1} loss {total/n:.4f}")

## 4. Evaluate on the test period

In [ ]:
model.eval(); preds = []
t0 = time.time()
with torch.no_grad():
    for i in range(0, len(yte), 64):
        batch = {k: v[i:i+64].to(device) for k, v in Xte.items()}
        preds.extend(model(**batch).logits.argmax(-1).cpu().numpy())
bert_time = time.time() - t0
preds = np.array(preds)
f1_bert = f1_score(yte, preds, average="macro")
print(classification_report(yte, preds, target_names=classes))
print("DistilBERT macro-F1:", round(f1_bert, 4))

## 5. Compare with the baseline

In [ ]:
base = joblib.load("../models/baseline.joblib")
tmp = test.copy()
tmp["hour"] = tmp.created_date.dt.hour.astype(str); tmp["weekday"] = tmp.created_date.dt.weekday.astype(str)
tmp["neighborhood"] = tmp["neighborhood"].fillna("Unknown")
t0 = time.time(); bp = base.predict(tmp); base_time = time.time() - t0
f1_base = f1_score(test["category"], bp, average="macro")
params = sum(p.numel() for p in model.parameters())
summary = pd.DataFrame({
    "model": ["TF-IDF + LogReg", "DistilBERT"],
    "macro_F1": [round(f1_base, 4), round(f1_bert, 4)],
    "inference_sec_total": [round(base_time, 2), round(bert_time, 2)],
    "approx_params": ["~ thousands (sparse)", f"{params/1e6:.0f}M"]})
summary

## 6. Save the model and results

In [ ]:
model.save_pretrained("../models/distilbert"); tok.save_pretrained("../models/distilbert")
summary.to_csv("../models/model_comparison.csv", index=False)

## 7. Findings (fill in)
- Macro-F1 baseline vs DistilBERT:
- Latency and size tradeoff:
- Which model would you ship, and why (cost, explainability, accuracy)?
- On real, messier text, expect DistilBERT to gain more than on this templated data.